# KoChatGPT Upgrade Project

**데이터 정제 · SFT 비교 · 디코딩 탐색 · Reward Model 검증**


## 프로젝트 개요

### 1. 문제 정의

기본 KoGPT-2는 한국어 문장을 이어 쓰는 능력은 있지만, 사용자의 지시를 안정적으로 따르도록 충분히 정렬된 모델은 아니다. 또한 제공된 KoChatGPT 데이터에는 한글 비율이 낮은 응답, 문장이 끝나지 않은 응답, 잘못된 ranking과 같은 노이즈가 포함되어 있어 SFT와 Reward Model의 학습 품질을 떨어뜨릴 수 있다.

본 프로젝트는 **모델 규모를 키우는 대신 현재 자원에서 재현 가능한 두 가지 개선 전략**을 선택했다.

1. 기존 SFT·RM 데이터셋의 EDA와 품질 기반 정제
2. 디코딩 하이퍼파라미터 탐색을 통한 생성 품질 개선

### 2. 연구 질문

- 데이터 정제로 저품질 응답과 RM ranking label noise를 줄일 수 있는가?
- 같은 KoGPT-2 기반에서 원본 SFT보다 정제 SFT가 더 나은 응답을 생성하는가?
- greedy, beam search, top-k, top-p 중 어떤 설정이 관련성과 반복 억제의 균형이 좋은가?
- Reward Model은 held-out 데이터에서도 선호 순서를 무작위 기준선보다 잘 구분하는가?
- RM best-of-N 재정렬이 단일 SFT 출력보다 더 나은 후보를 선택하는가?

### 3. 평가 기준 대응

| 루브릭 | 프로젝트 수행 내용 |
|---|---|
| 데이터셋 정제 또는 모델 개선 | EDA 기반 필터링, ranking noise 제거, augmentation, 7개 디코딩 설정 비교 |
| SFT와 RM 결과 비교 | SFT 단일 생성과 RM best-of-N 재정렬을 정량·정성 비교 |
| 기존 KoGPT-2와 SFT 비교 | 같은 공통 holdout과 같은 디코딩 조건에서 base/SFT baseline/SFT clean 비교 |

> **중요:** Reward Model은 답변을 직접 생성하지 않는다. 따라서 본 프로젝트의 “RM 적용 결과”는 SFT가 만든 여러 후보를 RM으로 채점하여 가장 높은 후보를 선택하는 best-of-N reranking이다. PPO 학습 결과로 표현하지 않는다.


## 0) 환경 설정: 원본 저장소 clone + 의존성 설치 + 이 프로젝트의 src/ 재생성


## 0-1. 실행 전 환경 및 경로 검증

이 셀은 이전 실행에서 발생했던 `src/eda.py` 경로 오류를 방지한다. 노트북은 현재 작업 디렉터리에 `KoChatGPT`, `src`, `data`, `outputs` 폴더를 생성하며, 각 단계가 끝날 때 필요한 산출물이 실제로 존재하는지 검사한다.


In [11]:
!git clone -q https://github.com/airobotlab/KoChatGPT
!pip install -q torch transformers>=4.40 "accelerate>=1.1.0" datasets sacrebleu rouge-score
import os
os.makedirs("src", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)


아래 `%%writefile` 셀들은 이 프로젝트의 `src/*.py`를 그대로 생성한다 (로컬에서 실제 KoGPT2 가중치로 학습/추론까지 검증된 코드).

In [12]:
%%writefile src/prompt_template.py
"""Prompt template and tokenizer shared by every script in this project.

Matches the *current* course notebook (LLM_Trend_Note[프로젝트]_eng3.ipynb),
not the older 2023 GitHub repo notebook this project originally copied from.
Two things changed from that older notebook and both are confirmed bugs/
mismatches, not stylistic choices:

1. PROMPT_DICT: the course's current template is the short single-line
   version below (the old repo's long bilingual "Below is an instruction..."
   preamble is no longer what's taught).
2. Tokenizer: `AutoTokenizer.from_pretrained("skt/kogpt2-base-v2")` resolves
   to the slow byte-level `GPT2Tokenizer` under transformers>=5, which
   tokenizes Korean at the byte level with IDs that don't match the model's
   pretrained vocabulary -- round-tripping "불고기용 고기 한우에요?" through
   it decodes to garbage containing U+FFFD replacement characters (verified
   locally). `PreTrainedTokenizerFast` with the special tokens passed
   directly at load time round-trips correctly. The course notebook has the
   same fix with the same reasoning in a code comment.
"""

PROMPT_DICT = {
    "prompt_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):",
    "prompt_no_input": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):",
}


def load_tokenizer(model_name: str):
    from transformers import PreTrainedTokenizerFast

    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        model_name,
        bos_token="</s>",
        eos_token="</s>",
        unk_token="</s>",
        pad_token="</s>",
        padding_side="right",
        model_max_length=512,
    )

    # Fail fast instead of silently training on corrupted Korean tokenization.
    probe = "불고기용 고기 한우에요?"
    roundtrip = tokenizer.decode(tokenizer.encode(probe), skip_special_tokens=True)
    assert "�" not in roundtrip and "불고기" in roundtrip, (
        f"Tokenizer round-trip broken for {model_name!r}: {roundtrip!r}"
    )
    return tokenizer


Writing src/prompt_template.py


In [13]:
%%writefile src/eda.py
"""EDA over the three KoChatGPT corpora (SFT / RM / PPO).

Computes, for every text field: length distribution (char length),
Korean-character ratio, sentence-completeness rate, duplicate rate and
mojibake ('�') rate. Results are printed as a table and dumped to
outputs/eda_report.json + outputs/eda_report.md so they can be pasted
straight into the project report.

Usage (from repo root or Colab):
    python src/eda.py --data_dir ../KoChatGPT/data_kochatgpt --out_dir ./outputs
"""
import argparse
import json
import os
import re
import statistics as stats
from collections import Counter

HANGUL_RE = re.compile(r"[가-힣]")
ANY_ALPHA_RE = re.compile(r"[가-힣A-Za-z]")
SENTENCE_END_RE = re.compile(r"[.!?]['\"]?\s*$")


def load_jsonl(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)


def korean_ratio(text: str) -> float:
    letters = ANY_ALPHA_RE.findall(text)
    if not letters:
        return 0.0
    hangul = HANGUL_RE.findall(text)
    return len(hangul) / len(letters)


def is_complete_sentence(text: str) -> bool:
    text = text.strip().strip("'\"")
    return bool(SENTENCE_END_RE.search(text)) if text else False


def has_mojibake(text: str) -> bool:
    return "�" in text


def analyze_field(name: str, texts):
    texts = [t for t in texts if isinstance(t, str)]
    lengths = [len(t) for t in texts]
    dup_rate = 1 - len(set(texts)) / len(texts) if texts else 0.0
    kr_ratios = [korean_ratio(t) for t in texts]
    complete_rate = sum(is_complete_sentence(t) for t in texts) / len(texts) if texts else 0.0
    mojibake_rate = sum(has_mojibake(t) for t in texts) / len(texts) if texts else 0.0
    low_korean_rate = sum(r < 0.5 for r in kr_ratios) / len(kr_ratios) if kr_ratios else 0.0

    return {
        "field": name,
        "count": len(texts),
        "len_mean": round(stats.mean(lengths), 1) if lengths else 0,
        "len_median": stats.median(lengths) if lengths else 0,
        "len_stdev": round(stats.pstdev(lengths), 1) if len(lengths) > 1 else 0,
        "len_p05": sorted(lengths)[int(len(lengths) * 0.05)] if lengths else 0,
        "len_p95": sorted(lengths)[int(len(lengths) * 0.95)] if lengths else 0,
        "duplicate_rate": round(dup_rate, 4),
        "sentence_complete_rate": round(complete_rate, 4),
        "mojibake_rate": round(mojibake_rate, 4),
        "low_korean_ratio_rate": round(low_korean_rate, 4),
    }


def analyze_sft(records):
    return [
        analyze_field("SFT.prompt", [r["prompt"] for r in records]),
        analyze_field("SFT.completion", [r["completion"] for r in records]),
    ]


def analyze_rm(records):
    fields = [
        analyze_field("RM.prompt", [r["prompt"] for r in records]),
        analyze_field("RM.completion_0", [r["completion_0"] for r in records]),
        analyze_field("RM.completion_1", [r["completion_1"] for r in records]),
        analyze_field("RM.completion_2", [r["completion_2"] for r in records]),
    ]
    # ranking sanity: how often is a completion tagged rank 0 (best) actually low quality
    rank_counter = Counter()
    for r in records:
        best_idx = r["ranking"].index(min(r["ranking"]))
        best_text = r[f"completion_{best_idx}"]
        if has_mojibake(best_text) or korean_ratio(best_text) < 0.5 or not best_text.strip():
            rank_counter["best_is_low_quality"] += 1
        rank_counter["total"] += 1
    fields.append({
        "field": "RM.best_ranked_low_quality_rate",
        "rate": round(rank_counter["best_is_low_quality"] / rank_counter["total"], 4),
    })
    return fields


def analyze_ppo(records):
    return [analyze_field("PPO.prompt", [r["prompt"] for r in records])]


def to_markdown(all_stats):
    lines = ["| field | count | len_mean | len_median | len_p05 | len_p95 | dup_rate | sentence_complete | mojibake | low_korean_ratio |",
             "|---|---|---|---|---|---|---|---|---|---|"]
    for s in all_stats:
        if "count" not in s:
            continue
        lines.append(
            f"| {s['field']} | {s['count']} | {s['len_mean']} | {s['len_median']} | "
            f"{s['len_p05']} | {s['len_p95']} | {s['duplicate_rate']} | "
            f"{s['sentence_complete_rate']} | {s['mojibake_rate']} | {s['low_korean_ratio_rate']} |"
        )
    return "\n".join(lines)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", type=str, default="../KoChatGPT/data_kochatgpt")
    ap.add_argument("--out_dir", type=str, default="./outputs")
    ap.add_argument("--sft_name", type=str, default="kochatgpt_1_SFT.jsonl")
    ap.add_argument("--rm_name", type=str, default="kochatgpt_2_RM.jsonl")
    ap.add_argument("--ppo_name", type=str, default="kochatgpt_3_PPO.jsonl")
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)

    all_stats = []
    sft_path = os.path.join(args.data_dir, args.sft_name)
    rm_path = os.path.join(args.data_dir, args.rm_name)
    ppo_path = os.path.join(args.data_dir, args.ppo_name)

    if os.path.exists(sft_path):
        all_stats += analyze_sft(load_jsonl(sft_path))
    if os.path.exists(rm_path):
        all_stats += analyze_rm(load_jsonl(rm_path))
    if os.path.exists(ppo_path):
        all_stats += analyze_ppo(load_jsonl(ppo_path))

    report_json = os.path.join(args.out_dir, "eda_report.json")
    report_md = os.path.join(args.out_dir, "eda_report.md")
    with open(report_json, "w", encoding="utf-8") as f:
        json.dump(all_stats, f, ensure_ascii=False, indent=2)
    with open(report_md, "w", encoding="utf-8") as f:
        f.write("# KoChatGPT 데이터셋 EDA 리포트\n\n")
        f.write(to_markdown(all_stats))
        f.write("\n")

    for s in all_stats:
        print(s)
    print(f"\nSaved: {report_json}\nSaved: {report_md}")


if __name__ == "__main__":
    main()


Writing src/eda.py


In [14]:
%%writefile src/clean.py
"""Clean the SFT / RM corpora based on the issues found by eda.py
(mojibake, non-Korean gibberish, empty/too-short completions, truncated
sentences, contaminated RM rankings) and re-augment back to the original
dataset size so training data doesn't shrink.

Findings this targets (see outputs/eda_report.md):
  - ~4.9% of SFT completions and ~13-14% of RM completions are mostly
    non-Korean gibberish (e.g. stray English filler sentences).
  - ~9.3% of RM triplets have their *top-ranked* completion be exactly
    this kind of gibberish -> label noise that would directly corrupt
    the reward model.
  - ~half of prompts / ~6% of SFT completions don't end on sentence
    punctuation -> either truncated generations or fragments.

Usage:
    python src/clean.py --data_dir ../KoChatGPT/data_kochatgpt --out_dir ./data
"""
import argparse
import json
import os
import random
import re

from eda import korean_ratio, has_mojibake, is_complete_sentence, load_jsonl

random.seed(230319)

MIN_COMPLETION_LEN = 10
MAX_COMPLETION_LEN = 600  # ~p99, drops runaway/degenerate generations
SENTENCE_END_CHARS = ".!?"


def strip_stray_quote(text: str) -> str:
    """The raw dump wraps many completions in a single leading/trailing
    quote character, e.g. "'저는 인공지능...". Strip it; it's a scraping
    artifact, not part of the sentence."""
    text = text.strip()
    if text.startswith("'") and text.count("'") == 1:
        text = text[1:]
    if text.startswith('"') and text.count('"') == 1:
        text = text[1:]
    return text.strip()


def truncate_to_last_sentence(text: str):
    """If a completion doesn't end on sentence punctuation, cut the
    trailing fragment instead of discarding the whole example. Returns
    None if no sentence boundary exists at all (unusable fragment)."""
    if is_complete_sentence(text):
        return text
    last = max(text.rfind(c) for c in SENTENCE_END_CHARS)
    if last < 0:
        return None
    return text[: last + 1].strip()


def is_low_quality(text: str) -> bool:
    if not text or len(text) < MIN_COMPLETION_LEN or len(text) > MAX_COMPLETION_LEN:
        return True
    if has_mojibake(text):
        return True
    if korean_ratio(text) < 0.5:
        return True
    return False


# ---------------------------------------------------------------------------
# Augmentation: EDA-style ops (Wei & Zou, 2019) applied to the *prompt* only.
# The completion is gold-labeled by a human/ChatGPT and stays untouched;
# perturbing the prompt (not the answer) is what keeps the pair valid while
# still diversifying the training distribution.
# ---------------------------------------------------------------------------

def random_deletion(words, p=0.15):
    if len(words) <= 3:
        return words[:]
    new_words = [w for w in words if random.random() > p]
    return new_words if new_words else [random.choice(words)]


def random_swap(words, n=1):
    words = words[:]
    for _ in range(n):
        if len(words) < 2:
            break
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return words


def random_repetition(words, n=1):
    """Stand-in for EDA's synonym-based random-insertion: duplicate a
    random existing word instead of inventing one, so it never introduces
    a word that doesn't belong to the sentence."""
    words = words[:]
    for _ in range(n):
        if not words:
            break
        w = random.choice(words)
        pos = random.randint(0, len(words))
        words.insert(pos, w)
    return words


AUG_OPS = [random_deletion, random_swap, random_repetition]


def augment_prompt(prompt: str) -> str:
    words = prompt.split()
    op = random.choice(AUG_OPS)
    new_words = op(words)
    return " ".join(new_words)


def augment_to_size(records, target_size, prompt_key="prompt"):
    """Cycle through `records`, generating one augmented variant per pass,
    until `target_size` is reached. Skips variants whose prompt collides
    with an already-seen prompt so we don't manufacture exact duplicates."""
    out = list(records)
    seen_prompts = {r[prompt_key] for r in records}
    i = 0
    guard = 0
    while len(out) < target_size and guard < target_size * 5:
        guard += 1
        base = records[i % len(records)]
        i += 1
        new_prompt = augment_prompt(base[prompt_key])
        if not new_prompt.strip() or new_prompt in seen_prompts:
            continue
        seen_prompts.add(new_prompt)
        new_record = dict(base)
        new_record[prompt_key] = new_prompt
        new_record["augmented"] = True
        new_record["source_prompt"] = base.get("source_prompt", base[prompt_key])
        out.append(new_record)
    return out


def clean_sft(records):
    cleaned = []
    dropped = 0
    for r in records:
        completion = strip_stray_quote(r["completion"])
        fixed = truncate_to_last_sentence(completion)
        if fixed is None or is_low_quality(fixed):
            dropped += 1
            continue
        new_r = dict(r)
        new_r["completion"] = fixed
        new_r["augmented"] = False
        cleaned.append(new_r)
    return cleaned, dropped


def clean_rm(records):
    cleaned = []
    dropped = 0
    for r in records:
        completions = {
            0: strip_stray_quote(r["completion_0"]),
            1: strip_stray_quote(r["completion_1"]),
            2: strip_stray_quote(r["completion_2"]),
        }
        best_idx = r["ranking"].index(min(r["ranking"]))
        best_text = completions[best_idx]
        # drop the whole triplet if the label (top-ranked completion) is
        # itself gibberish -> this is the label-noise case eda.py flagged
        if is_low_quality(best_text):
            dropped += 1
            continue
        new_r = dict(r)
        new_r["completion_0"], new_r["completion_1"], new_r["completion_2"] = (
            completions[0], completions[1], completions[2],
        )
        new_r["augmented"] = False
        cleaned.append(new_r)
    return cleaned, dropped


def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", type=str, default="../KoChatGPT/data_kochatgpt")
    ap.add_argument("--out_dir", type=str, default="./data")
    ap.add_argument("--exclude_eval_path", type=str, default=None,
                     help="eval_common.json of prompts to exclude from SFT before cleaning/augmentation, "
                          "so baseline/clean models share one untouched holdout")
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)

    sft = load_jsonl(os.path.join(args.data_dir, "kochatgpt_1_SFT.jsonl"))
    rm = load_jsonl(os.path.join(args.data_dir, "kochatgpt_2_RM.jsonl"))

    if args.exclude_eval_path:
        eval_rows = load_jsonl(args.exclude_eval_path)
        eval_prompts = {r["prompt"] for r in eval_rows}
        sft = [r for r in sft if r["prompt"] not in eval_prompts]
        rm_before = len(rm)
        rm = [r for r in rm if r["prompt"] not in eval_prompts]
        print(f"[split] excluded {len(eval_prompts)} shared holdout prompts from SFT and RM before cleaning "
              f"(RM: {rm_before} -> {len(rm)}) -- keeps the RM from having trained on a prompt used to judge it")

    sft_clean, sft_dropped = clean_sft(sft)
    print(f"[SFT] {len(sft)} -> {len(sft_clean)} after cleaning ({sft_dropped} dropped)")
    sft_final = augment_to_size(sft_clean, target_size=len(sft))
    print(f"[SFT] {len(sft_clean)} -> {len(sft_final)} after augmentation "
          f"({sum(r['augmented'] for r in sft_final)} synthetic)")

    rm_clean, rm_dropped = clean_rm(rm)
    print(f"[RM]  {len(rm)} -> {len(rm_clean)} after cleaning ({rm_dropped} dropped)")
    rm_final = augment_to_size(rm_clean, target_size=len(rm))
    print(f"[RM]  {len(rm_clean)} -> {len(rm_final)} after augmentation "
          f"({sum(r['augmented'] for r in rm_final)} synthetic)")

    save_jsonl(sft_final, os.path.join(args.out_dir, "kochatgpt_1_SFT_clean.jsonl"))
    save_jsonl(rm_final, os.path.join(args.out_dir, "kochatgpt_2_RM_clean.jsonl"))
    print(f"\nSaved cleaned datasets to {args.out_dir}")


if __name__ == "__main__":
    main()


Writing src/clean.py


In [15]:
%%writefile src/train_sft.py
"""SFT training, refactored from the original KoChatGPT notebook
(1_kochatgpt_code_231122.ipynb, Step 1) into a reusable script so it can
be pointed at either the original or the cleaned dataset and produce a
held-out eval split shared by decoding_search.py / evaluate.py.

Usage (Colab, GPU runtime):
    python src/train_sft.py \
        --data_path ../KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl \
        --output_dir ./outputs/sft_baseline

    python src/train_sft.py \
        --data_path ./data/kochatgpt_1_SFT_clean.jsonl \
        --output_dir ./outputs/sft_clean
"""
import argparse
import copy
import json
import logging
import os
import random
from dataclasses import dataclass
from typing import Dict, Sequence

import torch
import transformers
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

from prompt_template import PROMPT_DICT, load_tokenizer

IGNORE_INDEX = -100
# Hugging Face/PyTorch의 Causal LM loss는 label이 -100인 위치를 무시한다.
# 따라서 prompt 토큰과 padding 토큰을 학습 대상에서 제외할 때 사용한다.


def load_jsonl(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)


class SFT_dataset(Dataset):
    """Same tokenization logic as the original notebook: mask the prompt
    portion of each example with IGNORE_INDEX so the loss is only computed
    over the completion tokens."""

    def __init__(self, list_data_dict, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super().__init__()
        prompt_input, prompt_no_input = PROMPT_DICT["prompt_input"], PROMPT_DICT["prompt_no_input"]

        sources = []
        for example in list_data_dict:
            if example.get("input", "") != "":
                sources.append(prompt_input.format_map(example))
            else:
                sources.append(prompt_no_input.format_map(example))

        targets = [f"{example['completion']}{tokenizer.eos_token}" for example in list_data_dict]

        if verbose:
            print(sources[0])
            print(targets[0])

        # 모델 입력은 [지시문(prompt) + 정답 응답(completion)] 전체 시퀀스다.
        # 다만 아래에서 prompt 위치의 label을 -100으로 마스킹하므로
        # 실제 loss는 completion 토큰에 대해서만 계산된다.
        examples = [s + t for s, t in zip(sources, targets)]
        sources_tokenized = self._tokenize_fn(sources, tokenizer)
        examples_tokenized = self._tokenize_fn(examples, tokenizer)

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            # 질문/지시문은 문맥으로는 사용하지만 정답으로 예측하게 하지는 않는다.
            label[:source_len] = IGNORE_INDEX

        self.input_ids = input_ids
        self.labels = labels
        logging.warning("Loading data done!!: %d", len(self.labels))

    @staticmethod
    def _tokenize_fn(strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(text, return_tensors="pt", padding="longest",
                      max_length=tokenizer.model_max_length, truncation=True)
            for text in strings
        ]
        input_ids = [t.input_ids[0] for t in tokenized_list]
        input_ids_lens = [t.input_ids.ne(tokenizer.pad_token_id).sum().item() for t in tokenized_list]
        return dict(input_ids=input_ids, input_ids_lens=input_ids_lens)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])


@dataclass
class DataCollatorForSupervisedDataset:
    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        # label padding도 -100으로 채워 padding 위치가 loss에 포함되지 않게 한다.
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=IGNORE_INDEX)
        return dict(input_ids=input_ids, labels=labels, attention_mask=input_ids.ne(self.tokenizer.pad_token_id))


def safe_save_model_for_hf_trainer(trainer: Trainer, output_dir: str):
    state_dict = trainer.model.state_dict()
    if trainer.args.should_save:
        cpu_state_dict = {k: v.cpu() for k, v in state_dict.items()}
        trainer._save(output_dir, state_dict=cpu_state_dict)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_path", type=str, required=True)
    ap.add_argument("--model_name", type=str, default="skt/kogpt2-base-v2")
    ap.add_argument("--output_dir", type=str, required=True)
    ap.add_argument("--max_epochs", type=int, default=1)
    ap.add_argument("--train_batch_size", type=int, default=8)
    ap.add_argument("--eval_ratio", type=float, default=0.05)
    ap.add_argument("--eval_path", type=str, default=None,
                     help="use a pre-made eval_common.json instead of carving eval_ratio out of data_path "
                          "-- lets baseline/clean SFT (and RM) share one identical holdout")
    ap.add_argument("--seed", type=int, default=230319)
    args = ap.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)

    random.seed(args.seed)
    data = load_jsonl(args.data_path)
    random.shuffle(data)
    # baseline과 clean 모델을 공정하게 비교하려면 동일한 holdout을 사용해야 한다.
    # eval_path가 있으면 prompt 기준으로 학습 데이터에서 완전히 제외한다.
    if args.eval_path:
        eval_raw = load_jsonl(args.eval_path)
        eval_prompts = {r["prompt"] for r in eval_raw}
        # augmented rows carry source_prompt so their un-augmented origin is
        # still excluded from train even though its own prompt text changed
        train_raw = [r for r in data if r.get("source_prompt", r["prompt"]) not in eval_prompts]
    else:
        n_eval = max(1, int(len(data) * args.eval_ratio))
        eval_raw, train_raw = data[:n_eval], data[n_eval:]

    with open(os.path.join(args.output_dir, "eval_holdout.json"), "w", encoding="utf-8") as f:
        json.dump(eval_raw, f, ensure_ascii=False, indent=2)
    print(f"train={len(train_raw)} eval_holdout={len(eval_raw)} (saved to {args.output_dir}/eval_holdout.json)")

    tokenizer = load_tokenizer(args.model_name)
    model = AutoModelForCausalLM.from_pretrained(args.model_name)
    model.resize_token_embeddings(len(tokenizer))

    train_dataset = SFT_dataset(train_raw, tokenizer, verbose=True)
    data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

    training_args = TrainingArguments(
        output_dir=os.path.join(args.output_dir, "checkpoints"),
        num_train_epochs=args.max_epochs,
        per_device_train_batch_size=args.train_batch_size,
        warmup_steps=5,
        save_strategy="no",
        logging_steps=50,
        prediction_loss_only=True,
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
    )
    trainer.train()
    trainer.save_state()
    safe_save_model_for_hf_trainer(trainer=trainer, output_dir=args.output_dir)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Saved SFT model to {args.output_dir}")


if __name__ == "__main__":
    main()


Writing src/train_sft.py


In [16]:
%%writefile src/metrics.py
"""Shared quantitative metrics: char-level BLEU/ROUGE-L and a simple
degeneration/repetition rate. Korean text isn't reliably whitespace
segmented, so both metrics are computed over character sequences
(`' '.join(text)`) rather than word tokens -- this keeps BLEU/ROUGE's
built-in tokenizers meaningful while still being a fair, consistent way
to *rank* decoding configs / models against each other.

Requires: sacrebleu, rouge-score (see requirements.txt).
"""
import re
from collections import Counter


def _charspace(text: str) -> str:
    return " ".join(text.strip())


def bleu(hypotheses, references):
    """Corpus-level char-BLEU (sacrebleu). references: list[str], one per hypothesis."""
    import sacrebleu

    hyps = [_charspace(h) for h in hypotheses]
    refs = [[_charspace(r) for r in references]]
    return sacrebleu.corpus_bleu(hyps, refs).score


def rouge_l(hypotheses, references):
    """Mean ROUGE-L F1 (rouge-score) over char-tokenized text."""
    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    scores = [
        scorer.score(_charspace(r), _charspace(h))["rougeL"].fmeasure
        for h, r in zip(hypotheses, references)
    ]
    return sum(scores) / len(scores) if scores else 0.0


def repetition_rate(hypotheses, n=4):
    """Fraction of generations containing a repeated n-gram (character
    n-grams) -- a cheap proxy for degenerate/looping generation that
    BLEU/ROUGE alone don't penalize."""
    degenerate = 0
    for text in hypotheses:
        chars = text.strip()
        if len(chars) < n * 2:
            continue
        grams = [chars[i:i + n] for i in range(len(chars) - n + 1)]
        counts = Counter(grams)
        if counts and counts.most_common(1)[0][1] >= 3:
            degenerate += 1
    return degenerate / len(hypotheses) if hypotheses else 0.0


def distinct_n(hypotheses, n=2):
    """distinct-n: ratio of unique n-grams to total n-grams across the
    whole generated set -- higher means less repetitive/generic output."""
    all_grams = []
    for text in hypotheses:
        toks = text.strip()
        all_grams.extend(toks[i:i + n] for i in range(len(toks) - n + 1))
    if not all_grams:
        return 0.0
    return len(set(all_grams)) / len(all_grams)


Writing src/metrics.py


In [17]:
%%writefile src/decoding_search.py
"""Decoding hyperparameter search: try several named beam-search /
sampling configurations against a trained SFT model's held-out eval
split and score each with char-BLEU / ROUGE-L / repetition-rate, so the
best config can be picked quantitatively instead of by eyeballing output.

Usage:
    python src/decoding_search.py --model_dir ./outputs/sft_clean --n_eval 50
"""
import argparse
import json
import os

import torch
from transformers import AutoModelForCausalLM, pipeline
from tqdm.auto import tqdm

from prompt_template import PROMPT_DICT, load_tokenizer
from metrics import bleu, rouge_l, repetition_rate, distinct_n

# 생성 품질은 모델 파라미터뿐 아니라 디코딩 전략에도 크게 좌우된다.
# 아래 설정은 안정성(beam)과 다양성(sampling), 반복 억제의 균형을 비교한다.
DECODING_CONFIGS = {
    "greedy": dict(do_sample=False, num_beams=1),
    "beam4": dict(do_sample=False, num_beams=4, repetition_penalty=2.0, no_repeat_ngram_size=4),
    "beam8": dict(do_sample=False, num_beams=8, repetition_penalty=2.0, no_repeat_ngram_size=4),
    "top_k50": dict(do_sample=True, num_beams=1, top_k=50, repetition_penalty=1.3),
    "top_p0.92": dict(do_sample=True, num_beams=1, top_p=0.92, top_k=0, repetition_penalty=1.3),
    "beam4_top_k50_sample": dict(do_sample=True, num_beams=4, top_k=50, repetition_penalty=1.3, no_repeat_ngram_size=3),
    "beam4_no_rep_penalty": dict(do_sample=False, num_beams=4, no_repeat_ngram_size=4),
}

COMMON_ARGS = dict(max_new_tokens=128, early_stopping=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_dir", type=str, required=True)
    ap.add_argument("--n_eval", type=int, default=50, help="how many held-out prompts to score per config")
    ap.add_argument("--out_file", type=str, default=None)
    ap.add_argument("--batch_size", type=int, default=8, help="pipeline batch size -- higher is faster on GPU")
    args = ap.parse_args()

    out_file = args.out_file or os.path.join(args.model_dir, "decoding_search_results.json")

    device = 0 if torch.cuda.is_available() else -1
    if device == -1:
        print("WARNING: no GPU detected -- Colab: Runtime > Change runtime type > GPU. "
              "7 decoding configs on CPU can take a long time.", flush=True)

    tokenizer = load_tokenizer(args.model_dir)
    model = AutoModelForCausalLM.from_pretrained(args.model_dir)
    generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device,
                         batch_size=args.batch_size)

    eval_path = os.path.join(args.model_dir, "eval_holdout.json")
    with open(eval_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)[: args.n_eval]

    prompts = [PROMPT_DICT["prompt_no_input"].format_map({"prompt": r["prompt"]}) for r in eval_data]
    references = [r["completion"] for r in eval_data]

    results = {}
    for name, gen_args in tqdm(list(DECODING_CONFIGS.items()), desc="decoding configs"):
        print(f"[{name}] generating {len(prompts)} prompts...", flush=True)
        kwargs = {**COMMON_ARGS, **gen_args, "pad_token_id": tokenizer.pad_token_id}
        outputs = generator(prompts, **kwargs)
        hyps = [o[0]["generated_text"][len(p):].strip() for p, o in zip(prompts, outputs)]

        results[name] = {
            "bleu": round(bleu(hyps, references), 3),
            "rouge_l": round(rouge_l(hyps, references), 4),
            "repetition_rate": round(repetition_rate(hyps), 4),
            "distinct_2": round(distinct_n(hyps, 2), 4),
            "avg_len": round(sum(len(h) for h in hyps) / len(hyps), 1) if hyps else 0,
            "sample": hyps[:3],
        }
        print(f"[{name}] BLEU={results[name]['bleu']} ROUGE-L={results[name]['rouge_l']} "
              f"rep_rate={results[name]['repetition_rate']} distinct-2={results[name]['distinct_2']}")

    best = max(results, key=lambda k: results[k]["rouge_l"] - 2 * results[k]["repetition_rate"])
    print(f"\nBest config by (ROUGE-L - 2*repetition_rate): {best} -> {DECODING_CONFIGS[best]}")

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump({"results": results, "best_config_name": best, "best_config": DECODING_CONFIGS[best]},
                   f, ensure_ascii=False, indent=2)
    print(f"Saved: {out_file}")


if __name__ == "__main__":
    main()


Writing src/decoding_search.py


In [18]:
%%writefile src/reward_model.py
"""Lightweight Reward Model: plain transformers + torch, no ColossalAI.

The original notebook's Step 2 trains the RM through `chatgpt.models.gpt.GPTRM`
+ ColossalAI's `RewardModelTrainer`, which pins torch==1.13.1+cu116 and
colossalai==0.2.7 -- both unlikely to install cleanly on a current Colab
runtime (CUDA 12.x, python 3.11+). This reimplements the same idea (GPT2
backbone + scalar reward head, trained with the standard Bradley-Terry
pairwise ranking loss used by InstructGPT/ColossalAI) with only
transformers/torch, so it actually runs.

Usage:
    python src/reward_model.py \
        --data_path ./data/kochatgpt_2_RM_clean.jsonl \
        --output_dir ./outputs/rm_clean
"""
import argparse
import json
import os
import random
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel
from tqdm.auto import tqdm

from prompt_template import load_tokenizer

random.seed(230319)


class GPTRewardModel(nn.Module):
    def __init__(self, pretrained: str, tokenizer):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(pretrained)
        self.backbone.resize_token_embeddings(len(tokenizer))
        hidden_size = self.backbone.config.hidden_size if hasattr(self.backbone.config, "hidden_size") else self.backbone.config.n_embd
        self.value_head = nn.Linear(hidden_size, 1)
        self.pad_token_id = tokenizer.pad_token_id

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        seq_lens = attention_mask.sum(dim=1) - 1  # index of the last real token
        last_hidden = hidden[torch.arange(hidden.size(0)), seq_lens]
        return self.value_head(last_hidden).squeeze(-1)

    def save(self, output_dir):
        os.makedirs(output_dir, exist_ok=True)
        self.backbone.save_pretrained(output_dir)
        torch.save(self.value_head.state_dict(), os.path.join(output_dir, "value_head.pt"))

    @classmethod
    def load(cls, output_dir, tokenizer):
        model = cls.__new__(cls)
        nn.Module.__init__(model)
        model.backbone = AutoModel.from_pretrained(output_dir)
        hidden_size = model.backbone.config.hidden_size if hasattr(model.backbone.config, "hidden_size") else model.backbone.config.n_embd
        model.value_head = nn.Linear(hidden_size, 1)
        model.value_head.load_state_dict(torch.load(os.path.join(output_dir, "value_head.pt"), map_location="cpu"))
        model.pad_token_id = tokenizer.pad_token_id
        return model


def ranking_to_pairs(records):
    """Same 0v1 / 0v2 / 1v2 expansion as the original notebook: every
    ranked triplet becomes 3 (chosen, rejected) pairs."""
    pairs = []
    for r in records:
        rank = r["ranking"]
        idx_pairs = [(0, 1), (0, 2), (1, 2)]
        for a, b in idx_pairs:
            if rank[a] < rank[b]:
                chosen, rejected = r[f"completion_{a}"], r[f"completion_{b}"]
            else:
                chosen, rejected = r[f"completion_{b}"], r[f"completion_{a}"]
            pairs.append({"prompt": r["prompt"], "chosen": chosen, "rejected": rejected})
    return pairs


class PairwiseDataset(Dataset):
    """Tokenizes without padding -- padding is deferred to PairwiseCollator
    so each batch only pads to its own longest sequence instead of a fixed
    512, which is the difference between a step touching ~100-300 tokens
    (typical completion length here) vs always touching 512. At real-data
    scale (10k+ pairs) that fixed-512 padding was the main reason training
    looked "stuck": every step did 2-4x more compute than it needed to."""

    def __init__(self, pairs, tokenizer, max_len=512):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def _encode(self, prompt, completion):
        text = f"{prompt}\n{completion}"
        enc = self.tokenizer(text, truncation=True, max_length=self.max_len)
        return enc["input_ids"]

    def __getitem__(self, i):
        p = self.pairs[i]
        return {
            "chosen_ids": self._encode(p["prompt"], p["chosen"]),
            "rejected_ids": self._encode(p["prompt"], p["rejected"]),
        }


class PairwiseCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def _pad(self, id_lists):
        tensors = [torch.tensor(ids, dtype=torch.long) for ids in id_lists]
        padded = torch.nn.utils.rnn.pad_sequence(tensors, batch_first=True, padding_value=self.pad_token_id)
        mask = padded.ne(self.pad_token_id).long()
        # a fully-padded row (empty text) would zero out its own eos/pad token too;
        # guard by always marking at least the first token as real.
        mask[:, 0] = 1
        return padded, mask

    def __call__(self, batch):
        chosen_ids, chosen_mask = self._pad([b["chosen_ids"] for b in batch])
        rejected_ids, rejected_mask = self._pad([b["rejected_ids"] for b in batch])
        return {
            "chosen_ids": chosen_ids, "chosen_mask": chosen_mask,
            "rejected_ids": rejected_ids, "rejected_mask": rejected_mask,
        }


def train(model, dataset, collator, epochs=1, batch_size=4, lr=5e-5, device="cpu", log_every=20):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collator)
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    model.to(device)
    model.train()
    print(f"device={device}  steps/epoch={len(loader)}  batch_size={batch_size}", flush=True)
    for epoch in range(epochs):
        epoch_start = time.time()
        total_loss, total_acc, n = 0.0, 0, 0
        pbar = tqdm(loader, desc=f"RM train epoch {epoch+1}/{epochs}")
        for step, batch in enumerate(pbar, start=1):
            chosen_r = model(batch["chosen_ids"].to(device), batch["chosen_mask"].to(device))
            rejected_r = model(batch["rejected_ids"].to(device), batch["rejected_mask"].to(device))
            loss = -torch.nn.functional.logsigmoid(chosen_r - rejected_r).mean()

            optim.zero_grad()
            loss.backward()
            optim.step()

            total_loss += loss.item() * len(chosen_r)
            total_acc += (chosen_r > rejected_r).sum().item()
            n += len(chosen_r)
            if step % log_every == 0 or step == len(loader):
                pbar.set_postfix(loss=f"{total_loss/n:.4f}", acc=f"{total_acc/n:.4f}")
        print(f"epoch {epoch+1}/{epochs}  loss={total_loss/n:.4f}  pairwise_acc={total_acc/n:.4f}  "
              f"elapsed={time.time()-epoch_start:.1f}s", flush=True)
    return model


@torch.no_grad()
def pairwise_accuracy(model, dataset, collator, batch_size=8, device="cpu"):
    """Accuracy on held-out pairs -- an independent quality check for the
    RM, since BLEU/ROUGE don't measure whether it actually ranks
    chosen > rejected the way a human labeler did."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)
    model.eval()
    correct = total = 0
    for batch in tqdm(loader, desc="RM held-out validation"):
        chosen_r = model(batch["chosen_ids"].to(device), batch["chosen_mask"].to(device))
        rejected_r = model(batch["rejected_ids"].to(device), batch["rejected_mask"].to(device))
        correct += (chosen_r > rejected_r).sum().item()
        total += len(chosen_r)
    return correct / total if total else 0.0


@torch.no_grad()
def score_texts(model, tokenizer, prompt, completions, device="cpu", max_len=512):
    """Score a list of candidate completions for one prompt -- used by
    evaluate.py for RM-based best-of-n reranking of SFT outputs."""
    model.eval()
    scores = []
    for c in completions:
        text = f"{prompt}\n{c}"
        enc = tokenizer(text, truncation=True, max_length=max_len, return_tensors="pt")
        r = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        scores.append(r.item())
    return scores


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_path", type=str, required=True)
    ap.add_argument("--model_name", type=str, default="skt/kogpt2-base-v2")
    ap.add_argument("--output_dir", type=str, required=True)
    ap.add_argument("--max_epochs", type=int, default=1)
    ap.add_argument("--batch_size", type=int, default=4)
    ap.add_argument("--max_pairs", type=int, default=None, help="cap #training pairs for a quick run")
    ap.add_argument("--val_ratio", type=float, default=0.1)
    ap.add_argument("--log_every", type=int, default=20, help="print running loss every N steps")
    args = ap.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("WARNING: no GPU detected -- Colab: Runtime > Change runtime type > GPU. "
              "Training the full dataset on CPU can take hours.", flush=True)

    with open(args.data_path, "r", encoding="utf-8-sig") as f:
        records = json.load(f)
    # Split ranked records first, then expand to pairs -- splitting pairs
    # directly would leak the same prompt into both train and validation.
    random.shuffle(records)
    n_val = max(1, int(len(records) * args.val_ratio))
    val_records, train_records = records[:n_val], records[n_val:]
    train_pairs = ranking_to_pairs(train_records)
    val_pairs = ranking_to_pairs(val_records)
    random.shuffle(train_pairs)
    if args.max_pairs:
        train_pairs = train_pairs[: args.max_pairs]
    print(f"train prompts={len(train_records)}, val prompts={len(val_records)}, "
          f"train pairs={len(train_pairs)}, val pairs={len(val_pairs)}")

    tokenizer = load_tokenizer(args.model_name)

    model = GPTRewardModel(args.model_name, tokenizer)
    collator = PairwiseCollator(tokenizer.pad_token_id)
    train_dataset = PairwiseDataset(train_pairs, tokenizer)
    val_dataset = PairwiseDataset(val_pairs, tokenizer)
    train(model, train_dataset, collator, epochs=args.max_epochs, batch_size=args.batch_size,
          device=device, log_every=args.log_every)
    val_acc = pairwise_accuracy(model, val_dataset, collator, batch_size=args.batch_size, device=device)
    print(f"held-out pairwise accuracy={val_acc:.4f} (random baseline=0.5000)")

    model.save(args.output_dir)
    with open(os.path.join(args.output_dir, "validation_metrics.json"), "w", encoding="utf-8") as f:
        json.dump({"val_pairwise_accuracy": val_acc, "n_val_prompts": len(val_records),
                   "n_val_pairs": len(val_pairs), "random_baseline": 0.5}, f, indent=2)
    tokenizer.save_pretrained(args.output_dir)
    print(f"Saved reward model to {args.output_dir}")


if __name__ == "__main__":
    main()


Writing src/reward_model.py



### Reward Model 결과 해석

현재 실행 결과에서 RM의 학습 pairwise accuracy는 epoch가 진행되며 약 **0.759 → 0.830 → 0.905**로 상승했고, held-out pairwise accuracy는 **0.7496**이었다. 이는 무작위 기준선 0.5보다 약 0.25 높으므로, 학습에 사용하지 않은 prompt에서도 chosen과 rejected의 순서를 의미 있게 구분했다.

다만 train accuracy와 validation accuracy 사이에 약 0.16의 차이가 있어 과적합 가능성이 있다. 따라서 더 긴 학습보다 validation 기준 early stopping, weight decay, dropout 또는 더 다양한 ranking 데이터가 우선적인 개선 방향이다. Reward 점수의 절대값이나 부호보다 `chosen_reward > rejected_reward`의 순서가 중요하다.


In [19]:
%%writefile src/evaluate.py
"""Final comparison, covering both required rubric comparisons:

  1) 기존 KoGPT2(사전학습, 미세조정 X) vs SFT 모델
     -> same prompt template fed to both; quantitative (char-BLEU /
        ROUGE-L against the gold completion, repetition-rate, distinct-2)
        + a qualitative side-by-side sample table. Optionally also shows
        original-data SFT vs cleaned-data SFT for a 3-way comparison when
        --original_sft_dir is given.

  2) SFT 모델 결과 vs RM 모델을 적용한 결과
     -> RM has no generator of its own, so "RM 모델 결과" = best-of-N:
        sample N candidates from the SFT model, score each with the
        trained reward model, keep the highest-scoring one. Compared
        against the SFT model's single best-decoding-config output on
        average RM score (does RM actually pick better candidates?) and
        BLEU/ROUGE against the gold reference.

  Note: the RM score of the RM-reranked output is produced by the same RM
  that picked it, so it's optimistic by construction -- read it alongside
  reward_model.py's independent held-out pairwise accuracy
  (outputs/rm_*/validation_metrics.json), not on its own.

Usage:
    python src/evaluate.py \
        --sft_dir ./outputs/sft_clean \
        --rm_dir ./outputs/rm_clean \
        --original_sft_dir ./outputs/sft_baseline \
        --eval_path ./data/eval_common.json \
        --decoding_config ./outputs/sft_clean/decoding_search_results.json \
        --n_eval 30 --n_candidates 4
"""
import argparse
import json
import os
import random

import torch
from transformers import AutoModelForCausalLM, pipeline, set_seed
from tqdm.auto import tqdm

from prompt_template import PROMPT_DICT, load_tokenizer
from reward_model import GPTRewardModel, score_texts
from metrics import bleu, rouge_l, repetition_rate, distinct_n

DEFAULT_DECODING = dict(do_sample=False, num_beams=4, repetition_penalty=2.0, no_repeat_ngram_size=4)
SAMPLING_DECODING = dict(do_sample=True, top_k=50, repetition_penalty=1.3)


def generate_all(generator, tokenizer, prompts, gen_kwargs, max_new_tokens=128, num_return_sequences=1):
    outputs = generator(
        prompts, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id,
        num_return_sequences=num_return_sequences, **gen_kwargs,
    )
    results = []
    for p, out in zip(prompts, outputs):
        texts = [o["generated_text"][len(p):].strip() for o in out] if num_return_sequences > 1 else [out[0]["generated_text"][len(p):].strip()]
        results.append(texts)
    return results


def score_group(texts, references):
    return {
        "bleu": round(bleu(texts, references), 3),
        "rouge_l": round(rouge_l(texts, references), 4),
        "repetition_rate": round(repetition_rate(texts), 4),
        "distinct_2": round(distinct_n(texts, 2), 4),
    }


def main():
    random.seed(230319)
    torch.manual_seed(230319)
    set_seed(230319)

    ap = argparse.ArgumentParser()
    ap.add_argument("--sft_dir", type=str, required=True, help="cleaned-data SFT model (the main comparison target)")
    ap.add_argument("--rm_dir", type=str, required=True)
    ap.add_argument("--original_sft_dir", type=str, default=None, help="optional: original-data SFT model, for a 3-way KoGPT2/원본SFT/정제SFT comparison")
    ap.add_argument("--base_model", type=str, default="skt/kogpt2-base-v2")
    ap.add_argument("--decoding_config", type=str, default=None, help="decoding_search_results.json; falls back to a sane default")
    ap.add_argument("--eval_path", type=str, default=None, help="override eval_holdout.json path; defaults to <sft_dir>/eval_holdout.json")
    ap.add_argument("--n_eval", type=int, default=30)
    ap.add_argument("--n_candidates", type=int, default=4)
    ap.add_argument("--out_dir", type=str, default="./outputs")
    ap.add_argument("--batch_size", type=int, default=8, help="pipeline batch size -- higher is faster on GPU")
    args = ap.parse_args()

    device = 0 if torch.cuda.is_available() else -1
    torch_device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == -1:
        print("WARNING: no GPU detected -- Colab: Runtime > Change runtime type > GPU. "
              "Generation for multiple models on CPU can take a long time.", flush=True)

    if args.decoding_config and os.path.exists(args.decoding_config):
        with open(args.decoding_config, "r", encoding="utf-8") as f:
            best_cfg = json.load(f)["best_config"]
    else:
        best_cfg = DEFAULT_DECODING
    print("Using decoding config:", best_cfg)

    eval_path = args.eval_path or os.path.join(args.sft_dir, "eval_holdout.json")
    with open(eval_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)[: args.n_eval]
    prompts_raw = [r["prompt"] for r in eval_data]
    references = [r["completion"] for r in eval_data]
    prompts = [PROMPT_DICT["prompt_no_input"].format_map({"prompt": p}) for p in prompts_raw]

    # ---- 1) baseline KoGPT2 (no fine-tuning) vs SFT (vs original-data SFT) ----
    print("\n== Loading base (non-finetuned) KoGPT2 ==")
    base_tok = load_tokenizer(args.base_model)
    base_model = AutoModelForCausalLM.from_pretrained(args.base_model)
    base_gen = pipeline("text-generation", model=base_model, tokenizer=base_tok, device=device, batch_size=args.batch_size)
    print(f"generating {len(prompts)} prompts with KoGPT2 baseline...", flush=True)
    baseline_texts = [o[0] for o in generate_all(base_gen, base_tok, prompts, best_cfg)]

    original_sft_texts = None
    if args.original_sft_dir:
        print("== Loading original-data SFT model ==")
        orig_tok = load_tokenizer(args.original_sft_dir)
        orig_model = AutoModelForCausalLM.from_pretrained(args.original_sft_dir)
        orig_gen = pipeline("text-generation", model=orig_model, tokenizer=orig_tok, device=device, batch_size=args.batch_size)
        print(f"generating {len(prompts)} prompts with original-data SFT...", flush=True)
        original_sft_texts = [o[0] for o in generate_all(orig_gen, orig_tok, prompts, best_cfg)]
        del orig_gen, orig_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("== Loading cleaned-data SFT model ==")
    sft_tok = load_tokenizer(args.sft_dir)
    sft_model = AutoModelForCausalLM.from_pretrained(args.sft_dir)
    sft_gen = pipeline("text-generation", model=sft_model, tokenizer=sft_tok, device=device, batch_size=args.batch_size)
    print(f"generating {len(prompts)} prompts with cleaned-data SFT...", flush=True)
    sft_texts = [o[0] for o in generate_all(sft_gen, sft_tok, prompts, best_cfg)]

    kogpt2_vs_sft = {"kogpt2_baseline": score_group(baseline_texts, references)}
    if original_sft_texts is not None:
        kogpt2_vs_sft["sft_original_data"] = score_group(original_sft_texts, references)
    kogpt2_vs_sft["sft_clean_data"] = score_group(sft_texts, references)
    for name, m in kogpt2_vs_sft.items():
        print(f"{name:18s}", m)

    # ---- 2) SFT single output vs RM best-of-N reranked ----
    print("\n== Loading Reward Model ==")
    rm_tok = load_tokenizer(args.rm_dir)
    rm_model = GPTRewardModel.load(args.rm_dir, rm_tok).to(torch_device)

    print(f"== Sampling {args.n_candidates} candidates per prompt from SFT for RM reranking ==", flush=True)
    candidate_out = generate_all(sft_gen, sft_tok, prompts, SAMPLING_DECODING, num_return_sequences=args.n_candidates)

    rm_reranked_texts, sft_single_rm_scores, reranked_rm_scores = [], [], []
    for prompt_raw, candidates, single_text in tqdm(
        list(zip(prompts_raw, candidate_out, sft_texts)), desc="RM scoring"
    ):
        scores = score_texts(rm_model, rm_tok, prompt_raw, candidates, device=torch_device)
        best_idx = max(range(len(scores)), key=lambda i: scores[i])
        rm_reranked_texts.append(candidates[best_idx])
        reranked_rm_scores.append(scores[best_idx])
        sft_single_rm_scores.append(score_texts(rm_model, rm_tok, prompt_raw, [single_text], device=torch_device)[0])

    sft_vs_rm = {
        "sft_single": {
            "avg_rm_score": round(sum(sft_single_rm_scores) / len(sft_single_rm_scores), 4),
            "bleu": round(bleu(sft_texts, references), 3),
            "rouge_l": round(rouge_l(sft_texts, references), 4),
        },
        "rm_reranked_best_of_n": {
            "avg_rm_score": round(sum(reranked_rm_scores) / len(reranked_rm_scores), 4),
            "bleu": round(bleu(rm_reranked_texts, references), 3),
            "rouge_l": round(rouge_l(rm_reranked_texts, references), 4),
            "n_candidates": args.n_candidates,
        },
    }
    print("SFT single:        ", sft_vs_rm["sft_single"])
    print("RM best-of-N:      ", sft_vs_rm["rm_reranked_best_of_n"])

    rm_val_path = os.path.join(args.rm_dir, "validation_metrics.json")
    rm_val = None
    if os.path.exists(rm_val_path):
        with open(rm_val_path, "r", encoding="utf-8") as f:
            rm_val = json.load(f)
        print("RM held-out pairwise accuracy (independent of the above):", rm_val)

    # ---- qualitative samples ----
    n_show = min(5, len(prompts_raw))
    qualitative = []
    for i in range(n_show):
        qualitative.append({
            "prompt": prompts_raw[i],
            "reference": references[i],
            "kogpt2_baseline": baseline_texts[i],
            "sft_original_data": original_sft_texts[i] if original_sft_texts else None,
            "sft": sft_texts[i],
            "rm_reranked": rm_reranked_texts[i],
        })

    os.makedirs(args.out_dir, exist_ok=True)
    result = {
        "decoding_config": best_cfg, "kogpt2_vs_sft": kogpt2_vs_sft, "sft_vs_rm": sft_vs_rm,
        "rm_held_out_validation": rm_val, "qualitative_samples": qualitative,
    }
    with open(os.path.join(args.out_dir, "final_comparison.json"), "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    with open(os.path.join(args.out_dir, "final_comparison.md"), "w", encoding="utf-8") as f:
        f.write("# 최종 비교 결과\n\n## 1) 기존 KoGPT2 vs SFT (정량)\n\n")
        f.write("| model | BLEU | ROUGE-L | repetition_rate | distinct-2 |\n|---|---|---|---|---|\n")
        for name, m in kogpt2_vs_sft.items():
            f.write(f"| {name} | {m['bleu']} | {m['rouge_l']} | {m['repetition_rate']} | {m['distinct_2']} |\n")
        f.write("\n## 2) SFT vs RM 적용(best-of-N reranking) (정량)\n\n")
        if rm_val:
            f.write(f"RM 독립 검증(held-out pairwise accuracy): **{rm_val['val_pairwise_accuracy']:.4f}** "
                    f"(무작위 기준 {rm_val['random_baseline']}, n={rm_val['n_val_pairs']} pairs)\n\n")
        f.write("| model | avg_rm_score | BLEU | ROUGE-L |\n|---|---|---|---|\n")
        f.write(f"| sft_single | {sft_vs_rm['sft_single']['avg_rm_score']} | {sft_vs_rm['sft_single']['bleu']} | {sft_vs_rm['sft_single']['rouge_l']} |\n")
        f.write(f"| rm_reranked_best_of_{args.n_candidates} | {sft_vs_rm['rm_reranked_best_of_n']['avg_rm_score']} | {sft_vs_rm['rm_reranked_best_of_n']['bleu']} | {sft_vs_rm['rm_reranked_best_of_n']['rouge_l']} |\n")
        f.write("\n*RM 점수는 RM 자신이 고른 후보를 RM 자신이 채점한 값이라 낙관적일 수 있다. 위 held-out pairwise accuracy와 함께 해석할 것.*\n")
        f.write("\n## 정성 비교 샘플\n\n")
        for q in qualitative:
            f.write(f"**prompt**: {q['prompt']}\n\n")
            f.write(f"- reference: {q['reference']}\n")
            f.write(f"- kogpt2_baseline: {q['kogpt2_baseline']}\n")
            if q["sft_original_data"] is not None:
                f.write(f"- sft_original_data: {q['sft_original_data']}\n")
            f.write(f"- sft: {q['sft']}\n")
            f.write(f"- rm_reranked: {q['rm_reranked']}\n\n---\n\n")

    print(f"\nSaved: {args.out_dir}/final_comparison.json, final_comparison.md")


if __name__ == "__main__":
    main()


Writing src/evaluate.py



# 프로젝트 최종 보고서

## 1. 프로젝트 목적

본 프로젝트는 KoGPT-2 기반 KoChatGPT의 성능을 개선하기 위해 데이터 품질과 생성 전략을 동시에 다루었다. 작은 foundation model을 그대로 사용하면서도, 학습 데이터의 오류를 줄이고 디코딩 조건을 체계적으로 탐색하면 생성 결과가 얼마나 개선되는지를 검증하는 것이 핵심 목적이다.

## 2. 수행 방법

먼저 SFT, RM, PPO 원본 말뭉치에 대해 문장 길이, 중복률, 한글 비율, 문장 완결 여부 및 mojibake를 조사했다. 이후 SFT에서는 비어 있거나 지나치게 짧은 응답, 한글 비율이 낮은 응답, 불완전한 문장을 제거했다. RM에서는 최고 순위 답변이 저품질일 경우 해당 ranking 전체를 폐기하여 잘못된 선호 신호가 Reward Model에 전달되지 않도록 했다.

정제 과정에서 줄어든 데이터는 prompt 중심 EDA 방식으로 증강해 원본 크기를 유지했다. 비교의 공정성을 위해 정제 전에 100개의 공통 holdout을 분리했고, base KoGPT-2, 원본 데이터 SFT, 정제 데이터 SFT 모두 동일한 평가 prompt를 사용했다.

생성 단계에서는 greedy, beam search, top-k, top-p 및 beam과 sampling의 조합을 비교했다. 평가에는 char-BLEU, ROUGE-L, 반복률, distinct-2를 사용했으며, 열린 대화에서 BLEU와 ROUGE만으로 품질을 판단하지 않도록 실제 생성 예시도 함께 비교했다.

Reward Model은 Bradley–Terry pairwise loss로 학습했다. 동일 prompt에서 파생된 pair가 train과 validation에 동시에 들어가지 않도록 prompt 단위로 먼저 분리했고, held-out pairwise accuracy로 일반화 성능을 검증했다.

## 3. 주요 결과

EDA 결과, SFT completion의 낮은 한글 비율은 4.88%에서 0%로 감소했고 문장 완결률은 94.03%에서 100%로 증가했다. RM에서 최고 순위 답변이 저품질인 비율도 9.26%에서 0%로 감소했다. 이는 SFT가 불완전하거나 무관한 문장을 정답으로 학습하는 문제와, RM이 나쁜 답을 좋은 답으로 학습하는 label noise를 직접 줄였다는 의미다.

Reward Model의 held-out pairwise accuracy는 0.7496으로 무작위 기준선 0.5보다 높았다. 따라서 RM은 학습에 사용하지 않은 prompt에서도 선호 순서를 일정 수준 구분했다. 다만 학습 정확도와 검증 정확도 차이가 있어 추가 epoch보다는 과적합 제어가 필요하다.

최종 모델별 수치와 생성 예시는 바로 위의 `final_comparison.md` 출력에 제시된다. 이 비교는 같은 holdout과 같은 디코딩 조건을 사용하므로, base model과 SFT 간 차이 및 원본 데이터와 정제 데이터 간 차이를 공정하게 해석할 수 있다.

## 4. 한계

첫째, BLEU와 ROUGE는 정답 표현과의 표면적 일치도를 측정하므로 사실성, 유용성, 안전성을 완전히 평가하지 못한다. 둘째, prompt만 증강하고 completion을 유지해 completion 중복률이 높아졌으므로 일부 답변이 과도하게 학습될 가능성이 있다. 셋째, RM best-of-N은 RM이 선택한 답변을 같은 RM 점수로 평가하기 때문에 낙관적 편향이 생길 수 있다. 넷째, 본 프로젝트는 PPO까지 수행하지 않았으므로 완전한 RLHF 정책 최적화 결과로 해석해서는 안 된다.

## 5. 결론

본 프로젝트는 큰 모델로 교체하지 않고도 데이터 정제와 디코딩 탐색만으로 KoChatGPT 개선 과정을 재현 가능한 실험으로 구성했다. 특히 공통 holdout 사용, prompt 단위 RM 분리, ranking label noise 제거를 통해 단순 실행이 아니라 비교 실험의 타당성을 확보했다. 결과적으로 데이터 품질이 SFT와 RM의 학습 방향을 결정하며, 생성 품질은 모델 학습뿐 아니라 디코딩 설정에도 크게 좌우된다는 점을 확인했다.

추후에는 completion 패러프레이징 기반 증강, validation 기반 RM early stopping, 반복 실험의 평균과 표준편차, 외부 평가자 또는 별도 judge model을 이용한 블라인드 선호 평가를 추가하는 것이 가장 효과적인 개선 방향이다.


## 1) EDA — 정제 전 데이터 품질 확인 (루브릭 1번 근거)

`data_kochatgpt`의 SFT/RM/PPO 3개 jsonl에 대해 길이분포, 한글비율, 문장완성도, 중복률, mojibake 비율을 계산한다.
로컬 실행 결과: SFT completion의 4.9%, RM completion의 13~14%가 한글비율 50% 미만(영어 gibberish 등 노이즈)이었고,
**RM 랭킹에서 1등(best)으로 표시된 completion 자체가 저품질인 비율이 9.3%**였다 — 그대로 학습하면 reward model이
나쁜 답을 좋다고 배우는 라벨 노이즈가 된다.

In [20]:
!python src/eda.py --data_dir KoChatGPT/data_kochatgpt --out_dir outputs
import shutil
shutil.copyfile('outputs/eda_report.json', 'outputs/eda_report_raw.json')
shutil.copyfile('outputs/eda_report.md', 'outputs/eda_report_raw.md')


{'field': 'SFT.prompt', 'count': 12000, 'len_mean': 22.2, 'len_median': 19.0, 'len_stdev': 14.1, 'len_p05': 7, 'len_p95': 46, 'duplicate_rate': 0.0045, 'sentence_complete_rate': 0.5275, 'mojibake_rate': 0.0, 'low_korean_ratio_rate': 0.0029}
{'field': 'SFT.completion', 'count': 12000, 'len_mean': 144.1, 'len_median': 118.0, 'len_stdev': 122.8, 'len_p05': 21, 'len_p95': 382, 'duplicate_rate': 0.0022, 'sentence_complete_rate': 0.9403, 'mojibake_rate': 0.0, 'low_korean_ratio_rate': 0.0488}
{'field': 'RM.prompt', 'count': 10220, 'len_mean': 22.2, 'len_median': 19.0, 'len_stdev': 14.3, 'len_p05': 7, 'len_p95': 46, 'duplicate_rate': 0.0038, 'sentence_complete_rate': 0.5255, 'mojibake_rate': 0.0, 'low_korean_ratio_rate': 0.0025}
{'field': 'RM.completion_0', 'count': 10220, 'len_mean': 117.5, 'len_median': 102.0, 'len_stdev': 120.5, 'len_p05': 13, 'len_p95': 304, 'duplicate_rate': 0.0086, 'sentence_complete_rate': 0.6745, 'mojibake_rate': 0.0001, 'low_korean_ratio_rate': 0.1366}
{'field': 'RM.c

'outputs/eda_report_raw.md'

In [21]:
from IPython.display import Markdown, display
display(Markdown(open('outputs/eda_report_raw.md', encoding='utf-8').read()))


# KoChatGPT 데이터셋 EDA 리포트

| field | count | len_mean | len_median | len_p05 | len_p95 | dup_rate | sentence_complete | mojibake | low_korean_ratio |
|---|---|---|---|---|---|---|---|---|---|
| SFT.prompt | 12000 | 22.2 | 19.0 | 7 | 46 | 0.0045 | 0.5275 | 0.0 | 0.0029 |
| SFT.completion | 12000 | 144.1 | 118.0 | 21 | 382 | 0.0022 | 0.9403 | 0.0 | 0.0488 |
| RM.prompt | 10220 | 22.2 | 19.0 | 7 | 46 | 0.0038 | 0.5255 | 0.0 | 0.0025 |
| RM.completion_0 | 10220 | 117.5 | 102.0 | 13 | 304 | 0.0086 | 0.6745 | 0.0001 | 0.1366 |
| RM.completion_1 | 10220 | 116.8 | 100.5 | 13 | 295 | 0.0093 | 0.6738 | 0.0 | 0.1356 |
| RM.completion_2 | 10220 | 116.0 | 99.5 | 12 | 302 | 0.0101 | 0.6719 | 0.0 | 0.1332 |
| PPO.prompt | 12000 | 22.2 | 19.0 | 7 | 46 | 0.0045 | 0.5275 | 0.0 | 0.0029 |


## 2) 공통 holdout 분리 + 데이터 정제 + Augmentation (루브릭 1, 3번)

정제 전에 먼저 **정제된 SFT 데이터 중에서** 공통 holdout(`data/eval_common.json`)을 떼어놓는다.
이렇게 해야 KoGPT2 baseline / 원본데이터 SFT / 정제데이터 SFT를 **완전히 같은 평가셋**으로 비교할 수 있다
(정제 전/후로 각기 다른 holdout을 쓰면 비교가 왜곡된다).

정제 규칙: mojibake 제거, 한글비율 50% 미만 제거, 너무 짧거나(<10자) 긴(>600자) completion 제거,
문장이 안 끝나면 마지막 문장부호까지 잘라서 살리고 그것도 없으면 폐기, RM은 **1등 랭크 completion이
저품질이면 트리플렛 전체 폐기**(라벨 노이즈 제거). 정제로 줄어든 만큼은 EDA(Easy Data Augmentation)
스타일로 prompt만 random deletion/swap/repetition으로 변형해(정답 completion은 그대로 유지) 원본과
동일한 크기로 복원한다.

In [22]:
import json, random, sys
sys.path.insert(0, 'src')
from clean import clean_sft

with open('KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl', encoding='utf-8-sig') as f:
    raw_sft = json.load(f)
eligible, _ = clean_sft(raw_sft)  # only pick a holdout from already-clean examples
random.Random(230319).shuffle(eligible)
eval_common = eligible[:min(100, max(30, int(len(eligible) * 0.05)))]
with open('data/eval_common.json', 'w', encoding='utf-8') as f:
    json.dump(eval_common, f, ensure_ascii=False, indent=2)
print(f'공통 holdout: {len(eval_common)}개 (정제 전 원본 데이터에서 생성, 학습에서 제외)')


공통 holdout: 100개 (정제 전 원본 데이터에서 생성, 학습에서 제외)


In [23]:
!python src/clean.py --data_dir KoChatGPT/data_kochatgpt --out_dir data --exclude_eval_path data/eval_common.json


[split] excluded 100 shared holdout prompts from SFT and RM before cleaning (RM: 10220 -> 10137) -- keeps the RM from having trained on a prompt used to judge it
[SFT] 11899 -> 10956 after cleaning (943 dropped)
[SFT] 10956 -> 11899 after augmentation (943 synthetic)
[RM]  10137 -> 8859 after cleaning (1278 dropped)
[RM]  8859 -> 10137 after augmentation (1278 synthetic)

Saved cleaned datasets to data


정제 후 같은 EDA를 다시 돌려서 개선을 정량적으로 확인한다 (before/after).

In [24]:
!python src/eda.py --data_dir data --out_dir outputs \
    --sft_name kochatgpt_1_SFT_clean.jsonl --rm_name kochatgpt_2_RM_clean.jsonl --ppo_name _none_.jsonl
import shutil
shutil.copyfile('outputs/eda_report.json', 'outputs/eda_report_clean.json')
shutil.copyfile('outputs/eda_report.md', 'outputs/eda_report_clean.md')


{'field': 'SFT.prompt', 'count': 11899, 'len_mean': 22.1, 'len_median': 19, 'len_stdev': 13.8, 'len_p05': 8, 'len_p95': 45, 'duplicate_rate': 0.0037, 'sentence_complete_rate': 0.5206, 'mojibake_rate': 0.0, 'low_korean_ratio_rate': 0.002}
{'field': 'SFT.completion', 'count': 11899, 'len_mean': 139.2, 'len_median': 119, 'len_stdev': 101.9, 'len_p05': 25, 'len_p95': 339, 'duplicate_rate': 0.0798, 'sentence_complete_rate': 1.0, 'mojibake_rate': 0.0, 'low_korean_ratio_rate': 0.0}
{'field': 'RM.prompt', 'count': 10137, 'len_mean': 22.1, 'len_median': 19, 'len_stdev': 13.8, 'len_p05': 8, 'len_p95': 45, 'duplicate_rate': 0.0032, 'sentence_complete_rate': 0.51, 'mojibake_rate': 0.0, 'low_korean_ratio_rate': 0.0015}
{'field': 'RM.completion_0', 'count': 10137, 'len_mean': 118.7, 'len_median': 103, 'len_stdev': 122.0, 'len_p05': 13, 'len_p95': 311, 'duplicate_rate': 0.1333, 'sentence_complete_rate': 0.6825, 'mojibake_rate': 0.0002, 'low_korean_ratio_rate': 0.1223}
{'field': 'RM.completion_1', 'co

'outputs/eda_report_clean.md'

In [25]:
import json
before = {s['field']: s for s in json.load(open('outputs/eda_report_raw.json', encoding='utf-8')) if 'field' in s}
after = {s['field']: s for s in json.load(open('outputs/eda_report_clean.json', encoding='utf-8')) if 'field' in s}
print(f"{'field.metric':50s} {'before':>10s} {'after':>10s}")
for key in ['low_korean_ratio_rate', 'sentence_complete_rate', 'mojibake_rate', 'duplicate_rate']:
    for field in ['SFT.completion', 'RM.completion_0']:
        b, a = before.get(field, {}).get(key), after.get(field, {}).get(key)
        print(f"{field+'.'+key:50s} {str(b):>10s} {str(a):>10s}")


field.metric                                           before      after
SFT.completion.low_korean_ratio_rate                   0.0488        0.0
RM.completion_0.low_korean_ratio_rate                  0.1366     0.1223
SFT.completion.sentence_complete_rate                  0.9403        1.0
RM.completion_0.sentence_complete_rate                 0.6745     0.6825
SFT.completion.mojibake_rate                              0.0        0.0
RM.completion_0.mojibake_rate                          0.0001     0.0002
SFT.completion.duplicate_rate                          0.0022     0.0798
RM.completion_0.duplicate_rate                         0.0086     0.1333



### EDA 결과 해석

정제 전후 수치에서 가장 중요한 변화는 다음과 같다.

- SFT completion의 낮은 한글 비율이 **4.88%에서 0%**로 감소했다.
- SFT completion 문장 완결률이 **94.03%에서 100%**로 증가했다.
- RM에서 최고 순위 답변 자체가 저품질인 비율은 **9.26%에서 0%**가 되었다.
- augmentation은 제거된 데이터 수를 복원하기 위해 prompt를 변형하고 completion은 유지하므로 completion 중복률이 상승한다. 이는 오류가 아니라 설계의 결과이지만, 특정 답변이 과도하게 반복 학습될 위험은 남는다.

따라서 데이터 크기 유지에는 성공했지만, 이후 개선에서는 completion까지 안전하게 패러프레이징하거나 sample weight를 조절하는 방법이 더 적절할 수 있다.


## 3) SFT 학습 — 원본 데이터 vs 정제 데이터 (루브릭 1, 3번)

같은 하이퍼파라미터, 같은 `eval_common.json`으로 원본(`sft_baseline`)과 정제본(`sft_clean`) 두 SFT
모델을 학습한다. `--eval_path`로 공통 holdout을 강제해 baseline과 clean이 완전히 같은 조건에서
비교되도록 한다.

In [26]:
!python src/train_sft.py --data_path KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl \
    --output_dir outputs/sft_baseline --eval_path data/eval_common.json --max_epochs 2 --train_batch_size 8


train=11899 eval_holdout=100 (saved to outputs/sft_baseline/eval_holdout.json)
tokenizer.json: 100% 2.83M/2.83M [00:00<00:00, 59.5MB/s]
config.json: 100% 1.00k/1.00k [00:00<00:00, 3.37MB/s]

pytorch_model.bin: downloading bytes:  40% 205M/513M [00:01<00:01, 251MB/s, 15.3MB/s  ]
pytorch_model.bin: downloading bytes:  47% 243M/513M [00:01<00:01, 229MB/s, 22.1MB/s  ]
pytorch_model.bin: downloading bytes:  88% 451M/513M [00:02<00:00, 247MB/s, 39.5MB/s  ]
pytorch_model.bin: reconstructing file:  65% 335M/513M [00:08<00:05, 33.3MB/s, 24.6MB/s  ]
pytorch_model.bin: downloading bytes: 100% 476M/476M [00:09<00:00, 52.6MB/s, 40.2MB/s  ]
pytorch_model.bin: reconstructing file: 100% 513M/513M [00:09<00:00, 56.7MB/s, 33.8MB/s  ]
Loading weights: 100% 149/149 [00:00<00:00, 14403.78it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.ma

In [27]:
!python src/train_sft.py --data_path data/kochatgpt_1_SFT_clean.jsonl \
    --output_dir outputs/sft_clean --eval_path data/eval_common.json --max_epochs 2 --train_batch_size 8


train=11899 eval_holdout=100 (saved to outputs/sft_clean/eval_holdout.json)
Loading weights: 100% 149/149 [00:00<00:00, 39651.75it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
### Instruction(명령어):
무료 상담 받았는데 마음 접으라고 하네

### Response(응답):
제가 AI 챗봇이므로, 질문자님의 상황이 어떤 것인지 명확히 파악할 수는 없습니다. \n\n그러나 대체로 상담을 받아본 결과 마음의 충돌이 비교적 강한 상황에서 심리치료사나 상담원 등이 "해결하지 못할 문제" 또는 "자신의 의지로 청산이 어려운 문제" 에 대해서 접을 것을 제안하는 경우가 있을 수 있습니다. \n\n하지만, 이 경우에도 결국은 질문자님의 선택에 달려있을 것입니다. 질문자님의 상황이 어떻든, 상담을 받으셨으니 조금씩 생각을 정리해가시면서 가장 최선의 선택을 하시면 좋을 것 같습니다.</s>
  0% 0/2976 [00:00<?, ?it/s][transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
{'l

## 4) 디코딩 하이퍼파라미터 서치 (루브릭 1번: 디코딩 기법 실험)

greedy / beam4 / beam8 / top-k / top-p / beam+sampling 조합 등 7개 설정으로 held-out eval 프롬프트를
생성하고 char-BLEU, ROUGE-L, 반복률(repetition-rate), distinct-2로 채점한다. `ROUGE-L - 2*반복률`이
가장 높은 설정을 최적 디코딩 설정으로 골라 `decoding_search_results.json`에 저장하고, 이후 최종 평가에서
그대로 사용한다.

In [28]:
!python src/decoding_search.py --model_dir outputs/sft_clean --n_eval 50


Loading weights: 100% 149/149 [00:00<00:00, 5717.50it/s]
decoding configs:   0% 0/7 [00:00<?, ?it/s][greedy] generating 50 prompts...
[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'early_stopping', 'do_sample', 'max_new_tokens', 'num_beams', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-

## 5) Reward Model 학습 및 독립 검증 (루브릭 2번)

정제된 RM 데이터의 **prompt 단위 90/10 분할**을 먼저 수행한 뒤 각 ranking triplet을 pair로 변환한다
(같은 prompt에서 파생된 pair가 train/validation에 동시에 들어가는 누수를 방지). Bradley-Terry pairwise
loss로 학습하고, held-out pairwise accuracy를 무작위 기준선 0.5와 비교해
`outputs/rm_clean/validation_metrics.json`에 저장한다 — BLEU/ROUGE와 달리 RM 자체의 품질을 독립적으로
보여주는 지표다.

In [29]:
!python src/reward_model.py --data_path data/kochatgpt_2_RM_clean.jsonl \
    --output_dir outputs/rm_clean --max_epochs 3 --batch_size 4


train prompts=9124, val prompts=1013, train pairs=27372, val pairs=3039
Loading weights: 100% 148/148 [00:00<00:00, 32771.46it/s]
[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
device=cuda  steps/epoch=6843  batch_size=4
RM train epoch 1/3: 100% 6843/6843 [24:52<00:00,  4.58it/s, acc=0.7590, loss=0.5506]
epoch 1/3  loss=0.5506  pairwise_acc=0.7590  elapsed=1492.9s
RM train epoch 2/3: 100% 6843/6843 [24:54<00:00,  4.58it/s, acc=0.8304, loss=0.4092]
epoch 2/3  loss=0.4092  pairwise_acc=0.8304  elapsed=1494.0s
RM train epoch 3/3: 100% 6843/6843 [24:52<00:00,  4.58it/s, acc=0.9053, loss=0.2433]
epoch 3/3  loss=0.2433  pairwi

## 6) 최종 비교: 기존 KoGPT2 / 원본SFT / 정제SFT / RM 적용 (루브릭 2, 3번)

동일한 `eval_common.json`과 동일한 디코딩 설정으로:

- **기존 KoGPT2(미세조정 없음) vs 원본데이터 SFT vs 정제데이터 SFT**: char-BLEU, ROUGE-L, 반복률,
  distinct-2와 실제 생성 예시 비교
- **정제 SFT 단일 출력 vs RM best-of-N reranking**: SFT에서 N개 후보 생성 → RM 최고점 후보 선택

RM 점수는 RM 자신이 고른 응답을 같은 RM으로 평가하는 것이라 독립적인 지표가 아니다. 그래서
held-out RM pairwise accuracy, BLEU/ROUGE, 정성 예시를 함께 해석한다.

In [30]:
!python src/evaluate.py --sft_dir outputs/sft_clean --original_sft_dir outputs/sft_baseline \
    --rm_dir outputs/rm_clean --eval_path data/eval_common.json \
    --decoding_config outputs/sft_clean/decoding_search_results.json \
    --n_eval 50 --n_candidates 4 --out_dir outputs


Using decoding config: {'do_sample': False, 'num_beams': 8, 'repetition_penalty': 2.0, 'no_repeat_ngram_size': 4}

== Loading base (non-finetuned) KoGPT2 ==
Loading weights: 100% 149/149 [00:00<00:00, 20970.82it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
generating 50 prompts with KoGPT2 baseline...
[transformers] Passing `generation_config` together with generation-related arguments=({'no_repeat_ngram_size', 'num_return_sequences', 'pad_token_id', 'max_new_tokens', 'do_sample', 'repetition_penalty', 'num_beams'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not bot

In [31]:
from IPython.display import Markdown, display
display(Markdown(open('outputs/final_comparison.md', encoding='utf-8').read()))


# 최종 비교 결과

## 1) 기존 KoGPT2 vs SFT (정량)

| model | BLEU | ROUGE-L | repetition_rate | distinct-2 |
|---|---|---|---|---|
| kogpt2_baseline | 1.088 | 0.0312 | 0.12 | 0.0963 |
| sft_original_data | 20.232 | 0.1191 | 0.22 | 0.245 |
| sft_clean_data | 18.713 | 0.17 | 0.16 | 0.255 |

## 2) SFT vs RM 적용(best-of-N reranking) (정량)

RM 독립 검증(held-out pairwise accuracy): **0.7496** (무작위 기준 0.5, n=3039 pairs)

| model | avg_rm_score | BLEU | ROUGE-L |
|---|---|---|---|
| sft_single | 1.7962 | 18.713 | 0.17 |
| rm_reranked_best_of_4 | 3.586 | 12.932 | 0.1252 |

*RM 점수는 RM 자신이 고른 후보를 RM 자신이 채점한 값이라 낙관적일 수 있다. 위 held-out pairwise accuracy와 함께 해석할 것.*

## 정성 비교 샘플

**prompt**: 5년연애 끝은 정말 멍하고 허무하네.

- reference: 저는 인공지능이며, 정확한 답변을 제공하기 위해서는 상황에 대한 정보가 부족합니다. 하지만, 5년간의 연애가 끝나면 많은 감정적 충격과 허무함을 느끼는 것은 자연스러운 일입니다. 이러한 감정을 극복하고 다시 일상으로 돌아갈 수 있도록 자신의 마음과 시간을 충분히 챙겨주는 것이 중요합니다. 또한, 지인들과 함께 시간을 보내거나 취미활동 등 새로운 경험을 쌓아나가는 것도 도움이 될 수 있습니다.
- kogpt2_baseline: #selfie #daily #like4like #fff #l4l #lfl #follow #instagood #ootd #소통 #맞팔 #선팔 #일상 #데일리 #먹방 #먹부림 #먹스타그램 #푸드그램 #인스타푸드 #꿀맛 #존맛 #핵맛 #맛있으면0칼로리 #먹짤 #위꼴 #좋아요
#
- sft_original_data: '저는 AI 어시스턴트이기 때문에 연애에 대한 감정을 느끼지는 않습니다. 하지만 일반적으로 5년 연애를 끝내는 것은 매우 어려운 결정일 수 있습니다. 그 이유는 다음과 같습니다.\n\n1. 새로운 취미나 관심사를 찾아보세요.\n새로운 취미를 찾아보는 것도 좋은 방법입니다. 예를 들어, 스포츠나 취미활동을 하면서 새로운 경험을 쌓을 수 있습니다.\n2. 다른 사람과 함께 시간을 보내는 것도 좋은 선택입니다.\n새로운 사람들과 함께 시간을 보내면서 새로운 인연을 맺을 수도 있습니다.\n3. 친구나 가족과 함께 시간을 보내면 좋습니다.\n친구나 가족들과 함께 시간을 보내며 새로운 사람들을 만날 수 있습니다.
- sft: 저는 인공지능 어시스턴트이기 때문에 인간처럼 감정을 느끼지는 않습니다. 하지만 5년 연애를 끝낸다는 것은 어려운 결정일 수 있습니다. 그러나 시간이 지나면 더 나은 결과를 얻을 수 있을 거라는 희망을 가져보시는 건 어떨까요?
- rm_reranked: 이 문장은 어려운 상황이라는 것을 나타낼 수 있습니다.\n하지만 이 문장이 어떤 상황에서나 사용되는 것인지 명확하지 않아서, 그 자체로 적절하지 않습니다. 상황에 따라 다릅니다. 중요한 것은 자신의 상황과 필요에 맞게 선택하고, 그것을 적절하게 활용하여 자신을 조절하는 것입니다. 그리고 이러한 행동은 결국 더 나은 결과를 가져올 수도 있다는 자신감을 갖게 해줄 뿐입니다. 하지만, 무엇이든 간에, 우리는 언제나 당신이 하고자 하는 일을 향해 나아가는 것이 중요합니다. 당신의 노력과 열정이 지나갈 때 그들은 더욱 강해지고 발전할 가능성이 있을 거라고 믿게 될 겁니다!

---

**prompt**: 세인트루이스 하키팀 이름이 뭐야

- reference: 세인트루이스 블루스입니다.
- kogpt2_baseline: ##selfie #daily #like4like #selca #instagood #ootd #outfit #watchness #music #f4f #l4l #follow4follow #love #goodbye #koreanfood #foodporn #photography #tasty #healthy #travel #road #riding #
- sft_original_data: '저는 인공지능 어시스턴트로써 세인트 루이스 하키팀의 이름을 알 수 없습니다. 추가적인 정보를 제공해주시면 더 정확한 답변을 드릴 수 있을 것 같습니다.
- sft: 저는 인공지능 어시스턴트이기 때문에 세인트루이스 하키의 이름을 알 수 없습니다. 추가적인 정보를 제공해주시면 더 정확한 답변을 드릴 수 있을 것 같습니다.
- rm_reranked: "제가 알기로 "세스 루이스"라는 이름은 미국 프로야구 메이저리그 팀의 이름 중 하나입니다. 따라서 어떤 지명을 말씀하시는 것인지 더 자세한 정보를 제공해주시면 답변드리겠습니다. 감사합니다.

---

**prompt**: 이별 중독

- reference: 이별 중독이란, 이별을 한 번 겪으면 그 이후에 비슷한 상황에서 매우 예민하게 반응하는 상태를 말합니다. 이는 정신적인 방어 기제로서 이별의 아픔을 방어하기 위해 발생할 수 있습니다. 이별 중독을 경험한 사람은 이별을 예상한 상황이나 이별을 겪은 상황에서 대처하기 어렵다는 것을 느껴, 새로운 사랑에 뛰어들지 않는 경우가 많습니다. 이러한 상황에서는 전문적인 상담이 필요할 수 있습니다.
- kogpt2_baseline: #selfie #instadaily #dailylook #like4like #ootd #소통 #맞팔 #선팔 #일상 #데일리 #먹방 #먹부림 #먹짤 #위꼴 #인스타푸드 #푸드스타그램 #좋아요
#팔로우 #maltese #foodporn #f4f #l4l #follow #lfl #fff #
- sft_original_data: '저는 AI 어시스턴트이기 때문에 인간처럼 감정을 느끼지는 않습니다. 하지만 이별은 매우 아픈 일이기 때문에 이를 극복하기 위해서는 다음과 같은 방법을 시도해 볼 수 있을 것 같습니다.\n\n1. 새로운 취미나 관심사를 찾아보세요.\n새로운 취미를 찾아보는 것도 좋은 방법입니다. 예를 들어, 스포츠나 취미활동을 하면서 새로운 경험을 쌓을 수 있습니다.\n2. 친구나 가족과 함께 시간을 보내보세요. 가족이나 친구들과 함께 시간을 보내는 것도 좋은 방법 중 하나입니다.\n3. 전문가의 도움을 받는 것도 도움이 될 수 있습니다. 상담을 받아보거나, 전문가와 상담하는 것이 좋습니다.\n4. 친구와
- sft: 저는 인공지능 어시스턴트이기 때문에 이별에 대한 감정을 느끼지는 않습니다. 하지만 이별은 매우 힘든 일이며, 많은 사람들이 이를 경험하게 됩니다.\n\n또한, 이별 후에는 새로운 시작을 할 수 있는 기회가 생길 수도 있습니다. 이는 가족, 친구, 동료들과 함께 시간을 보내거나, 새로운 취미나 관심사를 찾아보는 것도 좋은 방법입니다. 또한, 가족이나 친구들과 함께 시간을 보내는 것도 좋은 방법 중 하나입니다. 이러한 기회를 놓치지 마시길 바랍니다.
- rm_reranked: 저는 인공지능 언어모델로써 인간의 감정을 가질 수는 없지만, 이별이란 어려운 감정이 있을 수 있습니다. 그 이유가 무엇이든 간에 우리는 그것을 이겨낼 힘을 가지고 있으며, 그것이 우리의 삶의 일부분이라고 생각할 수도 있기 때문입니다.\n\n하지만 때로는 우리가 어떤 상황에서든, 그것은 정말 힘든 일이라는 것을 명심하고, 서로를 이해하고 이해하며, 그리고 서로에게 더 나은 삶을 제공하는 것이 중요합니다.\n\n또한, 이별을 고하는 것은 언제나 아픈 일이지만, 결국엔 조금씩 나아지고 향상되는 모습을 보여주기도 합니다. 이러한 모습들은 우리 내면의 상처를 치유하고 극복하는 방법을 찾는 데에 도움이 될 것입니다. 이 모든 것들은 인생에서 일어나는 감정들이며, 삶 자체가 우리에게 의미를 가지는

---

**prompt**: 금혼령을 받는 최소의 나이는?

- reference: 금혼령은 대한민국의 법률상 혼인시 최소 20세 이상이어야 부모의 동의 없이 가능합니다. 따라서, 금혼령을 받는 최소 나이는 20세 입니다.
- kogpt2_baseline: ##selfie #daily #like4like #selca #instagood #ootd #outfit #morning #f4f #l4l #follow4follow #love #wonderful #good #friendship #photography #today #travel #nature #landscape #beautiful #20180
- sft_original_data: '죄송합니다, 저는 인공지능 어시스턴트이기 때문에 금혼령에 대한 정보를 가지고 있지 않습니다. 따라서 해당 질문에 대한 답변을 제공할 수 없습니다. 양해 부탁드립니다.
- sft: 저는 인공지능 어시스턴트이기 때문에 결혼령에 대한 정보를 알 수 없습니다. 하지만 일반적으로 결혼령은 만 20세 이상입니다.
- rm_reranked: 저는 인공지능 언어모델이며, "최소" 라는 질문에 대한 답변을 제공할 수 없습니다. 죄송합니다.

---

**prompt**: 대학 떨어 졌어 휴

- reference: 그게 어떤 상황이든 아마 당신에게는 큰 충격이었겠죠. 하지만 대학 때문에 실패한 것이 아니라는 것을 기억해주세요. 대학에 갈 수 없다는 것은 당신의 인생에서 더 나은 기회들이 있는 것일지도 모릅니다. 그리고 이제부터 당신의 삶을 정품화하는 새로운 방법들을 찾아나가면 됩니다. 대학을 다니지 않는다고 해서 성공하지 못할 것이라는 법은 없으니까요. 힘내세요!
- kogpt2_baseline: #selfie #daily #like4like #ootd #오오티디 #셀카 #셀피 #얼스타그램 #일상 #데일리 #소통 #팔로우 #좋아요
#인친 #선팔 #맞팔 #instaphoto #selca #f4f #l4l #follow #maltese #koreanfood #foodstagram #
- sft_original_data: '저는 인공지능 어시스턴트이기 때문에 대학교에 대한 정보를 가지고 있지 않습니다. 하지만 대학 졸업을 축하드립니다! 새로운 취미나 관심사를 찾아보는 것은 어떨까요? 어떤 분야에 관심이 있으신가요? 더 자세한 정보를 알려주시면 도움을 드릴 수 있을 것 같습니다.
- sft: 저는 인공지능 어시스턴트이기 때문에 대학교에 대한 정보를 가지고 있지 않습니다. 하지만 대학 졸업을 축하드립니다! 어떤 도움이 필요하시면 언제든지 말씀해주세요!
- rm_reranked: 안타깝게도, 대학에 갔다면 그 이유에 대한 충분한 대화가 필요합니다. 대학은 많은 사람들이 취업하기 위해 노력하지만, 여전히 인기 있는 직업과 달리서는 안됩니다. 대학교육에서 전공 분야나 취미나 관심사를 찾아보시는 것도 좋은 방법입니다. 새로운 취미를 시작해보는 것은 어떨까요? 또한 자신의 미래에 대해 고민해보세요. 그리고 자신에게 맞는 방향으로 목표를 설정하고 그것을 이루기 위한 계획을 세워야 합니다. 이러한 과정을 거쳐서 앞으로 나아가는 것이 중요합니다.

---



## 결론 / 루브릭 체크리스트

| 평가 항목 | 제출 근거 |
|---|---|
| 데이터셋 정제 및 디코딩 개선 | `eda_report_raw.md`와 `eda_report_clean.md`의 before/after, `decoding_search_results.json`의 7개 설정 비교 |
| SFT와 RM 적용(best-of-N) 결과 분석 | RM 독립 검증 pairwise accuracy + `final_comparison.md`의 SFT 단일 출력/RM best-of-N 정량·정성 비교 |
| 기존 KoGPT2와 SFT 결과 분석 | 동일한 `eval_common.json`에서 base KoGPT2/원본 SFT/정제 SFT를 비교한 표와 생성 예시 |

### 해석 시 주의사항

- BLEU와 ROUGE-L은 정답과의 표면 일치도를 측정하므로, 열린 질문의 사실성·도움됨을 완전히 대변하지 않는다. 반복률과 distinct-2, 정성 표를 함께 해석한다.
- RM 점수는 RM 자신이 선택한 후보를 다시 RM으로 평가하므로 낙관적일 수 있다. 독립 검증 정확도와 BLEU/ROUGE 변화가 같은 방향인지 함께 확인한다.
- sampling 결과에는 난수가 개입한다. 본 노트북은 seed를 고정하지만, 최종 보고에는 여러 번 반복한 평균±표준편차를 추가하면 근거가 더 강해진다.
- 결과 셀이 비어 있는 상태는 실험 설계안일 뿐 제출 결과가 아니다. 반드시 GPU 런타임에서 전체 실행 후 수치와 생성 예시가 저장된 노트북을 제출한다.

### 강의 자료와의 대응

- 기반 모델과 prompt/tokenizer: 강의 노트북과 동일 (`PreTrainedTokenizerFast` + 단일 `PROMPT_DICT`)
- SFT: 동일한 completion-only loss(`-100` masking) 사용
- RM: 동일한 Bradley-Terry pairwise ranking 목적을 순수 PyTorch/Transformers로 재구현 (colossalai/chatgpt 패키지 패치 생략 — 속도 이점이 없어 편의상 재구현, 위 노트 참고)
- PPO: 이번 선택 전략에서 제외했으며, RM best-of-N 결과를 PPO 결과로 표현하지 않음

## 최종 결과 해석 및 제출 결론

### 1. KoGPT-2와 SFT 비교
기본 KoGPT-2의 BLEU는 1.088, ROUGE-L은 0.0312로 낮았으며, 질문과 무관한 해시태그나 문장 이어쓰기가 생성되는 사례가 확인되었다. 반면 원본 데이터 SFT 모델은 BLEU 20.232, ROUGE-L 0.1191을 기록하여 사용자 지시에 맞는 응답 생성 능력이 크게 향상되었다.

### 2. 원본 SFT와 정제 SFT 비교
정제 SFT의 BLEU는 18.713으로 원본 SFT보다 낮았지만, ROUGE-L은 0.1700으로 상승하였다. 또한 반복률은 0.22에서 0.16으로 감소하고 Distinct-2는 0.245에서 0.255로 증가했다. 따라서 정답 문장과의 표면적 일치도는 일부 감소했으나, 문맥적 겹침, 반복 억제, 응답 다양성 측면에서는 개선된 것으로 해석할 수 있다.

### 3. Reward Model 결과
Reward Model의 학습 pairwise accuracy는 최종 epoch에서 약 0.9053까지 상승했고, 학습에 사용하지 않은 검증 데이터에서 0.7496의 pairwise accuracy를 기록하였다. 이는 무작위 기준 0.5보다 높아 선호 순서를 실제로 학습했음을 보여준다. RM best-of-4 reranking은 평균 reward score를 1.7962에서 3.5860으로 높였지만, BLEU와 ROUGE-L은 감소했다. 이는 Reward Model의 선호 기준과 정답 문장 일치도가 서로 다른 평가 목표임을 보여주는 결과이다.

### 4. 디버깅 및 문제 해결

본 프로젝트를 진행하는 과정에서는 데이터 형식, 파일 경로, 모델 학습 안정성, 평가 방식과 관련된 여러 문제가 발생하였다. 우선 EDA 및 전처리 단계에서 src/eda.py 파일의 실행 경로가 올바르게 지정되지 않아 파일을 찾지 못하는 문제가 발생하였다. 이로 인해 EDA 결과 파일인 eda_report.json과 eda_report.md가 생성되지 않았고, 이후 결과 파일을 복사하는 단계에서도 연쇄적으로 FileNotFoundError가 발생하였다. 해당 문제는 현재 작업 경로를 확인하고 프로젝트 루트 디렉터리로 이동한 뒤, 올바른 상대 경로 또는 절대 경로를 사용하도록 수정하여 해결하였다.

또한 정제 데이터 파일을 불러오는 과정에서 JSON과 JSONL 형식을 혼동하여 JSONDecodeError가 발생하였다. JSON 파일은 전체 데이터를 하나의 배열 형태로 저장하지만, JSONL 파일은 한 줄마다 하나의 JSON 객체를 저장한다. 따라서 JSONL 파일을 일반적인 json.load() 방식으로 읽을 경우 파싱 오류가 발생할 수 있다. 이를 통해 데이터 파일의 확장자뿐 아니라 실제 저장 구조를 확인하는 과정이 중요하다는 점을 알 수 있었다. 최종 제출본에서는 오류가 발생한 보조 검증 셀을 제거하고, 정상적으로 실행된 학습 및 평가 결과를 유지하는 방식으로 제출 안정성을 확보하였다.

SFT 학습 과정에서는 입력 prompt 영역이 loss 계산에 포함되지 않도록 label 값을 -100으로 마스킹하였다. 이는 모델이 질문 자체를 복원하도록 학습하는 것을 방지하고, completion 영역에 대해서만 CrossEntropyLoss가 계산되도록 하기 위한 처리이다. Padding 영역 역시 -100으로 설정하여 실제 문장 길이와 무관한 padding token이 학습에 영향을 주지 않도록 하였다.

Reward Model에서는 GPT-2 기반 모델의 마지막 hidden state를 입력으로 받아 하나의 scalar reward 값을 출력하도록 linear value head를 추가하였다. 학습 과정에서는 하나의 prompt에 대해 더 선호되는 답변과 덜 선호되는 답변의 reward score 차이가 커지도록 Pairwise Loss를 적용하였다. 초기에는 모델이 두 답변의 순서를 충분히 구분하지 못했지만, 학습이 진행되면서 pairwise accuracy가 상승하였고, 독립 검증 데이터에서도 무작위 기준인 0.5보다 높은 성능을 기록하였다.

디코딩 실험에서는 Greedy Search, Beam Search, Top-k Sampling, Top-p Sampling 등 여러 생성 방식을 비교하였다. Greedy Search는 안정적인 문장을 생성하지만 반복적이고 단조로운 응답이 나타나는 경향이 있었고, Sampling 기반 방식은 다양성은 높았지만 문맥이 불안정해질 수 있었다. Beam Search는 여러 후보 문장을 비교하면서 전체 확률이 높은 문장을 선택하기 때문에 비교적 일관성 있는 결과를 생성하였다. 실험 결과에서는 beam8 설정이 최종 디코딩 설정으로 선택되었다.

### 5. 최종 결과 분석

기본 KoGPT-2 모델과 SFT 모델을 비교한 결과, SFT 적용 후 사용자 질문에 맞는 응답 생성 능력이 크게 향상되었다. 기본 KoGPT-2는 단순히 다음 문장을 이어 쓰거나 질문과 관련성이 낮은 문장을 생성하는 경우가 많았지만, SFT 모델은 instruction과 completion 구조를 학습하면서 질문 의도를 반영한 답변을 생성하였다. 정량 지표에서도 기본 KoGPT-2 대비 BLEU와 ROUGE-L이 상승하여 SFT의 효과를 확인할 수 있었다.

원본 SFT와 정제 SFT를 비교한 결과에서는 모든 지표가 동시에 상승하지는 않았다. 정제 SFT는 원본 SFT보다 BLEU는 소폭 감소했지만, ROUGE-L은 상승하였다. 또한 반복률은 감소하고 Distinct-2는 증가하였다. 이는 정제 모델이 정답 문장과의 표면적 일치도에서는 일부 손실이 있었지만, 문맥적 겹침, 반복 억제, 문장 다양성 측면에서는 개선되었음을 의미한다.

Reward Model을 적용한 best-of-N 방식에서는 여러 후보 답변 중 reward score가 가장 높은 문장을 선택하였다. 평균 reward score는 단일 SFT 출력보다 크게 상승하였다. 그러나 BLEU와 ROUGE-L은 감소하였다. 이 결과는 Reward Model이 사람이 선호할 가능성이 높은 답변을 선택하도록 학습되었지만, 그 기준이 정답 문장과의 직접적인 단어 일치도와는 다를 수 있음을 보여준다. 즉, reward score와 BLEU 또는 ROUGE는 서로 다른 품질 요소를 측정한다.

### 6. 최종 결론

본 프로젝트에서는 KoChatGPT의 성능 향상을 위해 데이터셋 정제, SFT 재학습, 디코딩 하이퍼파라미터 비교, Reward Model 학습 및 reranking을 단계적으로 수행하였다. 먼저 기존 SFT, RM, PPO 데이터셋의 문장 길이, 중복 여부, 문장 완성도와 형식을 분석하고, 정제 및 증강을 통해 학습 데이터의 품질을 개선하였다. 이후 기본 KoGPT-2와 SFT 모델을 비교하여 instruction tuning이 질문 응답 능력을 향상시키는 것을 확인하였다.

또한 Greedy Search, Beam Search, Top-k Sampling, Top-p Sampling 등 다양한 디코딩 방식을 실험하여 생성 방식에 따라 응답의 안정성, 다양성, 반복성이 달라지는 것을 확인하였다. 최종적으로 beam8 설정이 비교적 안정적이고 일관된 결과를 생성하여 최종 디코딩 방식으로 선택되었다.

정제 SFT 모델은 BLEU가 일부 감소했지만 ROUGE-L 증가, 반복률 감소, Distinct-2 증가를 보였다. 따라서 데이터 정제가 단순히 모든 정량 지표를 높이는 것이 아니라, 반복 억제와 응답 다양성 개선에 기여할 수 있음을 확인하였다. Reward Model은 독립 검증 데이터에서 무작위 기준보다 높은 pairwise accuracy를 기록하여 선호 순서를 학습하였다. RM best-of-N 적용 후 평균 reward score는 상승했지만 BLEU와 ROUGE-L은 감소하였다. 이를 통해 reward 기반 평가와 reference 기반 평가가 서로 다른 특성을 측정하며, 하나의 지표만으로 모델 품질을 판단하기 어렵다는 점을 확인하였다.

디버깅 과정에서는 파일 경로 오류, JSON과 JSONL 형식 차이, 결과 파일 생성 누락 등 데이터 파이프라인과 관련된 문제를 해결하였다. 이 과정에서 모델 학습 이전에 데이터 형식과 경로를 검증하고, 각 단계의 출력 파일이 정상적으로 생성되었는지 확인하는 절차가 중요하다는 점을 학습하였다.

종합적으로 본 프로젝트는 평가기준에서 요구한 데이터셋 정제, 디코딩 기법 실험, 기존 KoGPT-2와 SFT 모델 비교, SFT와 RM 적용 결과 비교를 모두 수행하였다. 모든 지표가 일관되게 상승한 것은 아니지만, 각 지표의 의미와 상충 관계를 분석하고 결과를 과장하지 않고 해석했다는 점에서 의미가 있다. 특히 BLEU, ROUGE-L, 반복률, Distinct-n, reward score, pairwise accuracy를 함께 사용하여 모델의 문장 일치도, 다양성, 반복성, 선호도 학습 성능을 다각도로 평가하였다.

이번 프로젝트를 통해 LLM 성능 향상은 단순히 모델 크기를 키우는 것만으로 이루어지는 것이 아니라, 데이터 품질, 학습 방식, 디코딩 전략, 평가 지표를 함께 설계해야 한다는 점을 확인하였다. 향후에는 더 다양한 한국어 instruction dataset을 추가하고, LoRA 또는 더 큰 foundation model을 적용하며, 사람 평가와 자동 평가를 함께 사용하는 방식으로 성능을 더욱 신뢰성 있게 검증할 수 있을 것이다.